In [8]:
import yfinance as yf

In [9]:
import numpy as np

In [10]:
import pandas as pd

In [11]:

def get_financials(ticker):
    try:
        stock = yf.Ticker(ticker)

        pnl = stock.financials  # Income Statement
        bs = stock.balance_sheet  # Balance Sheet
        cf = stock.cashflow  # Cash Flow Statement

        # Concatenate financials into one DataFrame
        fs = pd.concat([pnl, bs, cf])

        if fs.empty:
            return None

        # Replace NaN values with None for better JSON serialization
        fs = fs.replace({np.nan: None})

        # Transpose data so that items are rows and dates are columns
        fs_transposed = fs.T
        fs_transposed['Ticker'] = ticker
        fs_transposed.reset_index(inplace=True)
        fs_transposed.rename(columns={'index': 'Date'}, inplace=True)
        fs_transposed['Date'] = pd.to_datetime(fs_transposed['Date']).dt.year

        # Sort the data by Date and then limit it to the latest 4 years
        fs_transposed = fs_transposed.sort_values(by='Date', ascending=False).head(4)  # Limiting to 4 years

        return fs_transposed

    except Exception as e:
        print(f"Error fetching financials for {ticker}: {e}")
        return None


In [33]:
def get_company_financials(ticker):
    if not ticker:
        print("No ticker provided")
        return

    try:
        financial_data = get_financials(ticker)
        if financial_data is None or financial_data.empty:
            print(f"No financial data found for the given ticker: {ticker}")
            return

        # Set pandas to display all columns or a limited number of columns
        pd.set_option('display.max_columns', 20)  # Set to 20 columns (adjust as needed)
        
        # Print the financial data
        # print(financial_data)
        return financial_data

    except Exception as e:
        print(f"Failed to fetch stock data: {str(e)}")

In [34]:
ticker = "AAPL"

In [35]:
financial_data = get_company_financials(ticker)

In [36]:
print(  financial_data [["Date", "EBITDA"]] )

   Date          EBITDA
0  2025  144748000000.0
1  2024  134661000000.0
2  2023  125820000000.0
3  2022  130541000000.0


In [65]:
def dcf_valuation(fcf, shares_outstanding, growth_rate=0.05, discount_rate=0.10, terminal_growth=0.02, years=5):
    """
    Performs a simple DCF valuation.
    """
    try:
        # Use the most recent FCF as base
        latest_fcf = fcf
        if latest_fcf <= 0:
            raise ValueError("Latest FCF is non-positive, DCF may not be meaningful.")

        # Project FCF for given years
        projected_fcfs = [latest_fcf * ((1 + growth_rate) ** year) for year in range(1, years + 1)]

        # Discount projected FCFs to present value
        discounted_fcfs = [fcf / ((1 + discount_rate) ** year) for year, fcf in enumerate(projected_fcfs, start=1)]

        # Terminal value using Gordon Growth Model
        terminal_value = (projected_fcfs[-1] * (1 + terminal_growth)) / (discount_rate - terminal_growth)
        discounted_terminal_value = terminal_value / ((1 + discount_rate) ** years)

        # Enterprise value
        enterprise_value = sum(discounted_fcfs) + discounted_terminal_value

        # Intrinsic value per share
        intrinsic_value_per_share = enterprise_value / shares_outstanding

        return intrinsic_value_per_share

    except Exception as e:
        print(f"Error in DCF calculation: {e}")
        return None




In [43]:
# ticker = "AAPL"  # Example: Apple Inc.
# fcf, shares, price = get_financial_data(ticker)
# print (fcf)
# if fcf is not None:
    intrinsic_value = dcf_valuation(fcf, shares, growth_rate=0.06, discount_rate=0.10, terminal_growth=0.025, years=5)
    if intrinsic_value:
        print(f"Ticker: {ticker}")
        print(f"Current Price: ${price:.2f}")
        print(f"Intrinsic Value (DCF): ${intrinsic_value:.2f}")
        print(f"Undervalued by: {((intrinsic_value - price) / price) * 100:.2f}%")

Error fetching data: Cash flow data not available.
None


In [44]:
ticker = "AAPL"  # Example: Apple Inc.

In [45]:
financial_data = get_company_financials(ticker)

In [46]:
print(  financial_data [["Date", "EBITDA"]] )

   Date          EBITDA
0  2025  144748000000.0
1  2024  134661000000.0
2  2023  125820000000.0
3  2022  130541000000.0


In [47]:
fcf = financial_data ["EBITDA"].iloc[0]

In [63]:
print (fcf)

144748000000.0


In [51]:
stock = yf.Ticker(ticker)

In [52]:
print (stock)

yfinance.Ticker object <AAPL>


In [56]:
 shares = getattr(ticker, "fast_info", {}).get("shares_outstanding", None)


In [59]:
dict = stock.get_info()

In [62]:
shares =  (dict['sharesOutstanding'])

In [68]:
growth_rate = 0.06

In [70]:
beta = stock.info['beta']

In [71]:
discount_rate = abs(beta) * 0.10

In [72]:
terminal_growth = 0.03

In [75]:
intrinsic_value = round( dcf_valuation(fcf, shares, growth_rate, discount_rate, terminal_growth, years=5) , 2)

In [76]:
print (intrinsic_value)

142.57


In [77]:
current_price = stock.info['currentPrice']

In [78]:
print (current_price)

278.12
